<a href="https://colab.research.google.com/github/j8768667-creator/-Chord-sequencer/blob/main/Qwen%20Image%202.1%20EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen-Image-2.1 Precision Image Editing

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

![Model](https://img.shields.io/badge/Diffusion-Qwen--Image--2.1_INT8-7952b3.svg)
![Text Encoder](https://img.shields.io/badge/Text_Encoder-Qwen3--VL_8B_INT8-blue.svg)
![Conditioning](https://img.shields.io/badge/Conditioning-Up_to_10_Reference_Images-blueviolet.svg)
![Acceleration](https://img.shields.io/badge/Acceleration-Triton_+_xFormers-brightgreen.svg)

---

### Architecture & Conditioning Mechanics
Qwen-Image-2.1 is an open-weight foundation model designed for instruction-based image editing and generation. It operates on a 32-layer Single-Stream Diffusion Transformer (DiT) architecture where text prompt tokens and visual latent tokens are concatenated and processed through shared self-attention layers.

- **4-Channel Alpha VAE**: Features a four-channel variational autoencoder capable of directly generating and modifying RGBA images with native transparency, bypassing separate foreground matting steps.
- **Slot 1 (`<image1>`)**: The primary target canvas. The model anchors its spatial layout, perspective, and untargeted regions to this image.
- **Slots 2 to 10 (`<image2>` to `<image10>`)**: Auxiliary reference donors. The model extracts specified attributes (garments, backgrounds, accessories, lighting, or facial identity) and synthesizes them into `<image1>`.
- **Single-Image Processing ($N = 1$)**: When only Slot 1 is populated, instructions operate directly without index tags (e.g., background removal, style transfer, or lighting shifts).
- **Computational Acceleration**: The pipeline utilizes INT8 ConvRot matrix quantization, native hardware FP16 VAE decoding, and automatic reference token downscaling to 768px to prevent quadratic attention expansion during sampling.

In [ ]:
# @title 1. Environment Configuration & Model Initialization
# @markdown Configures CUDA memory allocators, downloads INT8 models via aria2, and starts the backend execution daemon.

import os
import sys
import time
import urllib.request
import subprocess
import torch

print("═" * 65)
print("SYSTEM ENVIRONMENT & HARDWARE CONFIGURATION")
print("═" * 65)

if not torch.cuda.is_available():
    raise SystemError("GPU acceleration required. Go to Runtime > Change runtime type > GPU.")

# Configure CUDA memory allocator to eliminate fragmentation and memory transfer overhead
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"• Active GPU      : {gpu_name}")
print(f"• Allocated VRAM  : {vram_gb:.2f} GB")
print(f"• CUDA Allocator  : expandable_segments enabled")

%cd /content
if not os.path.exists("/content/ComfyUI"):
    print("• Initializing core runtime engine...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q -U aria2 pillow websocket-client flask flask-cors pinggy comfy-kitchen triton xformers

# Install helper nodes
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("rgthree-comfy"):
    !git clone -q https://github.com/rgthree/rgthree-comfy.git
%cd /content/ComfyUI

os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/input", exist_ok=True)

print("\nDownloading Qwen-Image-2.1 INT8 checkpoints...")
# 1. Diffusion Model (INT8 ConvRot ~7.26 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors" \
  -d /content/ComfyUI/models/diffusion_models -o qwen_image_2.1_int8_convrot.safetensors

# 2. Text Encoder & VLM (Qwen3-VL 8B INT8 ConvRot ~9.35 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors" \
  -d /content/ComfyUI/models/text_encoders -o qwen3vl_8b_int8_convrot.safetensors

# 3. 4-Channel VAE with Alpha Support (~676 MB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors" \
  -d /content/ComfyUI/models/vae -o qwen_image_2.1_vae_bf16.safetensors

# 4. Demonstration Assets
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/portrait_model_denim.png"
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/clothing_light_blue_denim_shirt.png"
!wget -q -nc -P /content/ComfyUI/input \
  "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/angry_broccoli.png"
!wget -q -nc -O /content/ComfyUI/input/scene_tokyo_street.png \
  "https://raw.githubusercontent.com/QwenLM/Qwen-Image/main/assets/demo_images/demo_bg.png" || true

# Terminate any prior sessions
!fuser -k 8188/tcp > /dev/null 2>&1

print("\nStarting accelerated backend engine...")
engine_log = open("/content/engine.log", "w")
proc = subprocess.Popen([
    sys.executable, "/content/ComfyUI/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--fp16-vae",
    "--preview-method", "none"
], stdout=engine_log, stderr=subprocess.STDOUT)

print("Connecting to backend engine...", end="")
for _ in range(90):
    if proc.poll() is not None:
        print("\n❌ Backend engine process terminated unexpectedly!")
        with open("/content/engine.log", "r") as f:
            print(f.read())
        raise RuntimeError("Backend engine crashed during boot. See log above.")

    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                print(" Ready!")
                break
    except Exception:
        time.sleep(1)
        print(".", end="")
else:
    print("\n❌ Boot timed out. Printing /content/engine.log:")
    with open("/content/engine.log", "r") as f:
        print(f.read())
    raise TimeoutError("Backend engine failed to boot within 90 seconds.")

print("✓ Initialization complete. Run Cell 2 to launch the editing interface.")

In [ ]:
# @title 2. Launch Interface & Public Tunnel
# @markdown Initializes the editing interface with dynamic multi-reference slot management and real-time step streaming.

import os
import sys
import json
import time
import glob
import uuid
import base64
import logging
import threading
import subprocess
import urllib.request
import websocket
from PIL import Image
from flask import Flask, request, jsonify, Response, send_from_directory
from flask_cors import CORS
from IPython.display import display, HTML

try:
    import pinggy
except ImportError:
    !pip install -q pinggy
    import pinggy

log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

app = Flask(__name__)
CORS(app)

def optimize_donor_image(filename, max_dim=768):
    """Downscales reference donors to 768px to prevent DiT sequence token explosion."""
    filepath = os.path.join('/content/ComfyUI/input', filename)
    if not os.path.exists(filepath):
        return filename
    try:
        with Image.open(filepath) as img:
            if max(img.size) > max_dim:
                img.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
                opt_name = f"opt_{max_dim}_{filename}"
                opt_path = os.path.join('/content/ComfyUI/input', opt_name)
                img.save(opt_path)
                return opt_name
    except Exception as e:
        print(f"[Warning] Failed to optimize donor: {e}", flush=True)
    return filename

HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en" class="dark">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Qwen-Image-2.1 Precision Editing</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
  <style>
    @keyframes rainbowFlow {
      0% { background-position: 0% 50%; }
      50% { background-position: 100% 50%; }
      100% { background-position: 0% 50%; }
    }
    .rainbow-text {
      background: linear-gradient(90deg, #ff007f, #7928ca, #0070f3, #00dfd8, #ff007f);
      background-size: 300% auto;
      -webkit-background-clip: text;
      -webkit-text-fill-color: transparent;
      animation: rainbowFlow 8s linear infinite;
    }
    .rainbow-btn {
      background: linear-gradient(90deg, #ff007f, #7928ca, #0070f3, #00dfd8, #ff007f);
      background-size: 300% auto;
      animation: rainbowFlow 6s linear infinite;
      transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    }
    .rainbow-btn:hover {
      box-shadow: 0 0 30px rgba(255, 0, 127, 0.5), 0 0 45px rgba(0, 223, 216, 0.3);
      transform: translateY(-1px);
    }
    .glass-panel {
      background: rgba(15, 18, 30, 0.6);
      backdrop-filter: blur(24px);
      -webkit-backdrop-filter: blur(24px);
      border: 1px solid rgba(255, 255, 255, 0.1);
      box-shadow: 0 12px 40px rgba(0, 0, 0, 0.5);
    }
    .glass-card {
      background: rgba(22, 27, 46, 0.55);
      backdrop-filter: blur(16px);
      -webkit-backdrop-filter: blur(16px);
      border: 1px solid rgba(255, 255, 255, 0.08);
      transition: all 0.3s ease;
    }
    .glass-card:hover {
      border-color: rgba(255, 255, 255, 0.25);
      box-shadow: 0 0 20px rgba(121, 40, 202, 0.25);
    }
    .checkerboard {
      background-image: linear-gradient(45deg, #1e293b 25%, transparent 25%), linear-gradient(-45deg, #1e293b 25%, transparent 25%), linear-gradient(45deg, transparent 75%, #1e293b 75%), linear-gradient(-45deg, transparent 75%, #1e293b 75%);
      background-size: 16px 16px;
      background-position: 0 0, 0 8px, 8px -8px, -8px 0px;
      background-color: #0f172a;
    }
  </style>
</head>
<body class="bg-[#06070d] text-slate-100 min-h-screen font-sans antialiased pb-12 relative overflow-x-hidden">
  <!-- AMBIENT GLOW ORBS -->
  <div class="fixed -top-40 -left-40 w-96 h-96 bg-gradient-to-r from-pink-500/20 to-purple-600/20 rounded-full blur-3xl pointer-events-none"></div>
  <div class="fixed top-1/2 -right-40 w-96 h-96 bg-gradient-to-r from-cyan-400/20 to-blue-600/20 rounded-full blur-3xl pointer-events-none"></div>
  <div class="fixed -bottom-40 left-1/3 w-96 h-96 bg-gradient-to-r from-amber-400/15 to-rose-500/15 rounded-full blur-3xl pointer-events-none"></div>

  <header class="border-b border-white/10 bg-slate-950/40 backdrop-blur-xl sticky top-0 z-50 px-6 py-4 flex items-center justify-between">
    <div class="flex items-center space-x-3">
      <div class="w-10 h-10 rounded-2xl rainbow-btn flex items-center justify-center shadow-lg shadow-pink-500/20">
        <i class="fa-solid fa-wand-magic-sparkles text-white text-lg"></i>
      </div>
      <div>
        <h1 class="text-xl font-extrabold rainbow-text tracking-tight">Qwen-Image-2.1</h1>
        <p class="text-xs text-slate-400 font-medium">Brought to you by AI With Chucky</p>
      </div>
    </div>
    <div class="flex items-center space-x-3">
      <span class="px-3 py-1 bg-emerald-500/10 border border-emerald-500/30 text-emerald-400 text-xs rounded-full flex items-center space-x-1.5 backdrop-blur-md">
        <span class="w-2 h-2 rounded-full bg-emerald-400 animate-pulse"></span>
        <span>Engine Online</span>
      </span>
      <span id="slot_counter_badge" class="px-3 py-1 bg-purple-500/10 border border-purple-500/30 text-purple-300 text-xs rounded-full font-mono backdrop-blur-md">2 / 10 Slots Active</span>
    </div>
  </header>

  <main class="max-w-7xl mx-auto px-6 py-8 grid grid-cols-1 lg:grid-cols-12 gap-8 relative z-10">
    <!-- LEFT COLUMN: CONTROLS (5 Cols) -->
    <div class="lg:col-span-5 space-y-6">
      <div class="glass-panel rounded-3xl p-6 space-y-6">

        <!-- CONDITIONING SLOTS HEADER -->
        <div>
          <div class="flex justify-between items-center mb-3">
            <label class="text-xs font-bold uppercase tracking-wider text-slate-300 flex items-center space-x-2">
              <i class="fa-solid fa-layer-group text-cyan-400"></i>
              <span>Conditioning Slots (1 to 10)</span>
            </label>
            <button type="button" id="btn_add_slot" onclick="addDonorSlot()" class="text-xs font-semibold px-3 py-1 rounded-lg bg-white/5 hover:bg-white/10 text-cyan-300 border border-cyan-500/30 transition flex items-center space-x-1">
              <i class="fa-solid fa-plus text-[10px]"></i>
              <span>Add Donor</span>
            </button>
          </div>

          <!-- SLOTS CONTAINER -->
          <div class="grid grid-cols-2 gap-3 max-h-[360px] overflow-y-auto pr-1" id="slots_grid">
            <!-- Rendered dynamically via JS -->
          </div>
        </div>

        <!-- EDITING DIRECTIVE -->
        <div class="space-y-2">
          <div class="flex justify-between items-center">
            <label class="text-xs font-bold uppercase tracking-wider text-slate-300 flex items-center space-x-2">
              <i class="fa-solid fa-pen-nib text-pink-400"></i>
              <span>Directive Instruction</span>
            </label>
            <span class="text-[10px] text-cyan-300 font-mono" id="tag_hint">&lt;image1&gt; Canvas | &lt;image2&gt; Donor</span>
          </div>
          <textarea id="prompt_input" rows="4" placeholder="Type your instruction freely... e.g. Put the jacket from <image2> on the person in <image1>, keep the face and pose intact." class="w-full rounded-2xl bg-black/40 border border-white/10 p-3.5 text-xs text-slate-100 focus:outline-none focus:border-pink-500 transition resize-none placeholder-slate-500"></textarea>

          <!-- QUICK HELPER CHIPS -->
          <div class="flex flex-wrap gap-1.5 pt-1">
            <button type="button" onclick="insertTag('<image1>')" class="px-2 py-0.5 rounded-md bg-white/5 hover:bg-white/10 text-slate-300 border border-white/10 text-[10px] font-mono transition">+ &lt;image1&gt;</button>
            <button type="button" onclick="insertTag('<image2>')" class="px-2 py-0.5 rounded-md bg-white/5 hover:bg-white/10 text-slate-300 border border-white/10 text-[10px] font-mono transition">+ &lt;image2&gt;</button>
            <button type="button" onclick="insertPhrase('Keep the face, pose, and background intact.')" class="px-2 py-0.5 rounded-md bg-white/5 hover:bg-white/10 text-slate-300 border border-white/10 text-[10px] transition">+ Preserve Face/Pose</button>
            <button type="button" onclick="insertPhrase('Remove the background, and output a PNG image with transparent alpha background')" class="px-2 py-0.5 rounded-md bg-white/5 hover:bg-white/10 text-slate-300 border border-white/10 text-[10px] transition">+ Transparent PNG</button>
          </div>
        </div>

        <!-- PARAMETER SLIDERS -->
        <div class="space-y-4 pt-3 border-t border-white/10 text-xs">
          <div>
            <div class="flex justify-between text-slate-300 mb-1">
              <span class="font-medium">Sampling Steps</span>
              <span id="steps_val" class="text-cyan-400 font-mono font-bold">20</span>
            </div>
            <input type="range" id="steps_slider" min="4" max="40" value="20" oninput="document.getElementById('steps_val').innerText = this.value" class="w-full accent-cyan-400">
          </div>
          <div>
            <div class="flex justify-between text-slate-300 mb-1">
              <span class="font-medium">Guidance CFG Scale</span>
              <span id="cfg_val" class="text-pink-400 font-mono font-bold">1.0</span>
            </div>
            <input type="range" id="cfg_slider" min="1.0" max="3.0" step="0.1" value="1.0" oninput="document.getElementById('cfg_val').innerText = parseFloat(this.value).toFixed(1)" class="w-full accent-pink-500">
          </div>
          <div class="grid grid-cols-2 gap-3">
            <div>
              <label class="block text-slate-400 mb-1 font-medium">Resolution Budget</label>
              <select id="res_budget" class="w-full bg-slate-900/80 border border-white/10 rounded-xl p-2.5 text-slate-200 focus:outline-none focus:border-cyan-400 text-xs">
                <option value="1024" selected>1 MP Budget (Fast)</option>
                <option value="0">Native Source Aspect</option>
                <option value="2048">Native 2K Output</option>
              </select>
            </div>
            <div>
              <label class="block text-slate-400 mb-1 font-medium">Denoise Intensity</label>
              <input type="number" id="denoise_input" value="1.0" step="0.05" min="0.4" max="1.0" class="w-full bg-slate-900/80 border border-white/10 rounded-xl p-2 text-slate-200 focus:outline-none focus:border-cyan-400 text-xs">
            </div>
          </div>
        </div>

        <!-- DISPATCH BUTTON -->
        <button id="btn_generate" onclick="runGeneration()" class="w-full py-4 rounded-2xl rainbow-btn text-white font-bold tracking-wide flex items-center justify-center space-x-2 text-sm shadow-xl active:scale-[0.99] cursor-pointer">
          <i class="fa-solid fa-bolt"></i>
          <span>Render Edit Composition</span>
        </button>
      </div>
    </div>

    <!-- RIGHT COLUMN: VIEWPORT (7 Cols) -->
    <div class="lg:col-span-7 space-y-6">
      <!-- LIVE PROGRESS PANEL -->
      <div id="progress_card" class="hidden glass-panel rounded-3xl p-4">
        <div class="flex justify-between items-center text-xs mb-2">
          <span id="progress_title" class="font-bold rainbow-text flex items-center space-x-2">
            <i class="fa-solid fa-circle-notch fa-spin text-pink-400"></i>
            <span>Conditioning Diffusion Engine...</span>
          </span>
          <span id="progress_pct" class="font-mono font-bold text-cyan-300">0%</span>
        </div>
        <div class="h-2 bg-slate-800 rounded-full overflow-hidden">
          <div id="progress_bar" class="h-full rainbow-btn w-0 transition-all duration-200"></div>
        </div>
      </div>

      <!-- STAGE CONTAINER -->
      <div class="glass-panel rounded-3xl p-6 space-y-4">
        <div class="flex justify-between items-center">
          <h3 class="text-sm font-bold text-slate-200 flex items-center space-x-2">
            <i class="fa-solid fa-shapes text-pink-400"></i>
            <span>Conditioners & Composition Stage</span>
          </h3>
          <div class="flex items-center space-x-2">
            <span id="alpha_badge" class="hidden px-3 py-0.5 rounded-full text-xs font-mono bg-purple-500/20 text-purple-300 border border-purple-500/30">RGBA Alpha</span>
            <span id="timer_badge" class="hidden px-3 py-0.5 rounded-full text-xs font-mono bg-emerald-500/20 text-emerald-400 border border-emerald-500/30 font-bold">Finished</span>
          </div>
        </div>

        <!-- ACTIVE REFERENCES ROW -->
        <div class="flex items-center space-x-3 overflow-x-auto pb-2" id="stage_strip">
          <!-- Rendered dynamically -->
        </div>

        <!-- RESULT IMAGE VIEWPORT -->
        <div class="bg-black/50 border border-white/10 rounded-2xl p-3 text-center relative min-h-[420px] flex items-center justify-center overflow-hidden">
          <div id="result_placeholder" class="text-center space-y-3 p-8">
            <div class="w-16 h-16 mx-auto rounded-3xl bg-white/5 border border-white/10 flex items-center justify-center text-slate-600 text-3xl">
              <i class="fa-regular fa-image"></i>
            </div>
            <p class="text-xs text-slate-400 max-w-sm">Upload your images, describe your directive freely on the left, and click Render Edit Composition.</p>
          </div>
          <img id="stage_result" class="hidden max-h-[520px] w-auto mx-auto object-contain rounded-xl shadow-2xl transition-all duration-300">
        </div>

        <div id="download_bar" class="hidden flex justify-end pt-2">
          <a id="btn_download" download="qwen_composition.png" class="px-5 py-2.5 rounded-xl bg-white/10 hover:bg-white/20 text-slate-100 text-xs font-bold flex items-center space-x-2 border border-white/20 transition active:scale-95">
            <i class="fa-solid fa-download text-cyan-400"></i>
            <span>Download High-Res PNG</span>
          </a>
        </div>
      </div>
    </div>
  </main>

  <script>
    let slots = [
      { id: 1, type: 'canvas', filename: 'portrait_model_denim.png' },
      { id: 2, type: 'donor', filename: 'clothing_light_blue_denim_shirt.png' }
    ];

    function renderSlotsUI() {
      const grid = document.getElementById('slots_grid');
      const strip = document.getElementById('stage_strip');
      grid.innerHTML = '';
      strip.innerHTML = '';

      slots.forEach((s, idx) => {
        const slotNum = idx + 1;
        const isCanvas = slotNum === 1;
        const title = isCanvas ? 'Canvas &lt;image1&gt;' : `Donor ${slotNum - 1} &lt;image${slotNum}&gt;`;
        const demoBtn = isCanvas
          ? '<button onclick="useDemo(0)" class="text-cyan-400 hover:underline text-[10px]">Demo</button>'
          : `<button onclick="removeSlot(${idx})" class="text-rose-400 hover:underline text-[10px]">Remove</button>`;

        grid.innerHTML += `
          <div class="glass-card rounded-2xl p-2.5">
            <div class="flex justify-between items-center mb-1.5 text-[11px]">
              <span class="font-bold ${isCanvas ? 'rainbow-text' : 'text-slate-300'}">${title}</span>
              ${demoBtn}
            </div>
            <div onclick="document.getElementById('input_slot_${idx}').click()" class="border-2 border-dashed border-white/10 hover:border-pink-500/50 rounded-xl p-1 text-center cursor-pointer transition relative bg-black/40 overflow-hidden h-28 flex flex-col items-center justify-center">
              <img id="thumb_slot_${idx}" src="/input/${s.filename}" class="absolute inset-0 w-full h-full object-cover rounded-lg">
              <div class="relative z-10 bg-black/70 px-2 py-0.5 rounded-md text-[9px] text-slate-200 backdrop-blur-xs"><i class="fa-solid fa-upload"></i> Upload</div>
            </div>
            <input type="file" id="input_slot_${idx}" class="hidden" accept="image/*" onchange="uploadSlotFile(${idx}, this.files[0])">
          </div>
        `;

        strip.innerHTML += `
          <div class="w-20 shrink-0 glass-card rounded-xl p-1 text-center text-[9px] text-slate-400">
            <span class="font-medium">&lt;${slotNum}&gt; ${isCanvas ? 'Canvas' : 'Donor'}</span>
            <img src="/input/${s.filename}" class="w-full aspect-square object-cover rounded-lg mt-1">
          </div>
        `;
      });

      const count = slots.length;
      document.getElementById('slot_counter_badge').innerText = `${count} / 10 Slots Active`;
      document.getElementById('btn_add_slot').style.display = count >= 10 ? 'none' : 'inline-flex';

      if (count === 1) {
        document.getElementById('tag_hint').innerText = 'Single-Image Mode (No tags needed)';
      } else if (count === 2) {
        document.getElementById('tag_hint').innerText = '<image1> Canvas | <image2> Donor';
      } else {
        document.getElementById('tag_hint').innerText = `<image1> Canvas | <image2> to <image${count}> Donors`;
      }
    }

    function addDonorSlot(customFile = 'clothing_light_blue_denim_shirt.png') {
      if (slots.length >= 10) return alert('Maximum 10 conditioning slots supported by the model.');
      slots.push({
        id: slots.length + 1,
        type: 'donor',
        filename: customFile
      });
      renderSlotsUI();
    }

    function removeSlot(idx) {
      if (slots.length <= 1) return;
      slots.splice(idx, 1);
      renderSlotsUI();
    }

    function useDemo(slotIdx) {
      slots[0].filename = 'portrait_model_denim.png';
      renderSlotsUI();
    }

    function insertTag(tag) {
      const textarea = document.getElementById('prompt_input');
      textarea.value += (textarea.value.length ? ' ' : '') + tag;
      textarea.focus();
    }

    function insertPhrase(phrase) {
      const textarea = document.getElementById('prompt_input');
      textarea.value += (textarea.value.length ? ' ' : '') + phrase;
      textarea.focus();
    }

    async function uploadSlotFile(slotIdx, file) {
      if (!file) return;
      const fd = new FormData();
      fd.append('file', file);
      const res = await fetch('/api/upload', { method: 'POST', body: fd });
      const data = await res.json();
      slots[slotIdx].filename = data.filename;
      renderSlotsUI();
    }

    async function runGeneration() {
      const btn = document.getElementById('btn_generate');
      const progCard = document.getElementById('progress_card');
      const progBar = document.getElementById('progress_bar');
      const progTitle = document.getElementById('progress_title');
      const progPct = document.getElementById('progress_pct');
      const timerBadge = document.getElementById('timer_badge');
      const alphaBadge = document.getElementById('alpha_badge');
      const resImg = document.getElementById('stage_result');
      const placeholder = document.getElementById('result_placeholder');
      const dlBar = document.getElementById('download_bar');
      const dlBtn = document.getElementById('btn_download');

      const activeSlots = slots.map(s => s.filename);
      const instructionText = document.getElementById('prompt_input').value.trim();
      if (!instructionText) {
        alert('Please enter an editing directive.');
        return;
      }

      btn.disabled = true;
      btn.classList.add('opacity-50', 'cursor-not-allowed');
      progCard.classList.remove('hidden');
      timerBadge.classList.add('hidden');
      alphaBadge.classList.add('hidden');
      progBar.style.width = '5%';
      progPct.innerText = '5%';

      const payload = {
        instruction: instructionText,
        slots: activeSlots,
        steps: parseInt(document.getElementById('steps_slider').value),
        denoise: parseFloat(document.getElementById('denoise_input').value),
        cfg: parseFloat(document.getElementById('cfg_slider').value),
        resolution: parseInt(document.getElementById('res_budget').value)
      };

      const sse = new EventSource('/api/generate_stream?' + new URLSearchParams({ payload: JSON.stringify(payload) }));
      sse.onmessage = function(e) {
        const ev = JSON.parse(e.data);
        if (ev.type === 'progress') {
          const pct = Math.round((ev.value / ev.max) * 85) + 10;
          progBar.style.width = pct + '%';
          progPct.innerText = pct + '%';
          progTitle.innerHTML = `<i class="fa-solid fa-gear fa-spin text-cyan-400"></i><span>Sampling: Step ${ev.value}/${ev.max}</span>`;
        } else if (ev.type === 'executing') {
          progTitle.innerHTML = `<i class="fa-solid fa-microchip text-pink-400"></i><span>Executing: ${ev.node}</span>`;
        } else if (ev.type === 'done') {
          sse.close();
          progBar.style.width = '100%';
          progPct.innerText = '100%';
          progTitle.innerHTML = `<i class="fa-solid fa-check text-emerald-400"></i><span>Composition Complete!</span>`;
          btn.disabled = false;
          btn.classList.remove('opacity-50', 'cursor-not-allowed');
          timerBadge.innerText = ev.duration.toFixed(1) + 's';
          timerBadge.classList.remove('hidden');

          placeholder.classList.add('hidden');
          resImg.src = 'data:image/png;base64,' + ev.image_b64;
          resImg.classList.remove('hidden');
          dlBtn.href = 'data:image/png;base64,' + ev.image_b64;
          dlBar.classList.remove('hidden');

          if (ev.has_alpha) {
            resImg.parentElement.classList.add('checkerboard');
            alphaBadge.classList.remove('hidden');
          } else {
            resImg.parentElement.classList.remove('checkerboard');
          }
        }
      };
      sse.onerror = function() {
        sse.close();
        btn.disabled = false;
        btn.classList.remove('opacity-50', 'cursor-not-allowed');
      };
    }

    document.getElementById('prompt_input').value = 'Put the light blue denim shirt from <image2> on the person in <image1>, keep the face and pose intact.';
    renderSlotsUI();
  </script>
</body>
</html>'''

@app.route('/')
def index():
    return HTML_TEMPLATE

@app.route('/input/<path:filename>')
def serve_input(filename):
    return send_from_directory('/content/ComfyUI/input', filename)

@app.route('/api/upload', methods=['POST'])
def upload_endpoint():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'}), 400
    f = request.files['file']
    fname = f"{uuid.uuid4().hex[:8]}_{f.filename}"
    f.save(os.path.join('/content/ComfyUI/input', fname))
    print(f"\n[Storage] 📤 Asset Saved: {fname}", flush=True)
    return jsonify({'filename': fname})

@app.route('/api/generate_stream')
def generate_stream():
    params = json.loads(request.args.get('payload'))
    client_id = str(uuid.uuid4())
    raw_slots = params.get('slots', ['portrait_model_denim.png'])[:10]
    num_slots = len(raw_slots)

    # Slot 1 (Canvas) stays full-resolution. Slots 2..10 (Donors) get capped to 768px
    # to shrink sequence length and eliminate quadratic latency overhead
    optimized_slots = []
    for idx, slot_file in enumerate(raw_slots, 1):
        if idx == 1:
            optimized_slots.append(slot_file)
        else:
            optimized_slots.append(optimize_donor_image(slot_file, max_dim=768))

    print("\n" + "═" * 65, flush=True)
    print(f"[Execution] 📥 Task Received ({num_slots} Active Slots)", flush=True)
    print(f"• Directive : {params['instruction'][:85]}...", flush=True)
    for idx, s in enumerate(optimized_slots, 1):
        tag_role = 'Canvas (Full Res)' if idx == 1 else f'Donor {idx - 1} (768px Opt)'
        print(f"• Slot {idx} ({tag_role}): {s} (<image{idx}>)", flush=True)
    print(f"• Config    : Steps={params['steps']} | CFG={params['cfg']} | Denoise={params['denoise']} | Budget={params['resolution']}", flush=True)
    print("═" * 65, flush=True)

    # Deterministic workflow architecture: reuses hashed nodes in VRAM cache
    workflow = {
      "461": {"inputs": {"filename_prefix": "Qwen_image_2.1", "format": "png", "format.bit_depth": "8-bit", "format.input_color_space": "sRGB", "images": ["459:457", 0]}, "class_type": "SaveImageAdvanced"},
      "459:451": {"inputs": {"unet_name": "qwen_image_2.1_int8_convrot.safetensors", "weight_dtype": "default"}, "class_type": "UNETLoader"},
      "459:453": {"inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image", "device": "default"}, "class_type": "CLIPLoader"},
      "459:454": {"inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}, "class_type": "VAELoader"},
      "459:456": {"inputs": {"width": 1024, "height": 1024, "batch_size": 1}, "class_type": "EmptyLatentImage"},
      "459:457": {"inputs": {"samples": ["459:458", 0], "vae": ["459:454", 0]}, "class_type": "VAEDecode"},
      "459:458": {
        "inputs": {
          "seed": int(time.time() * 1000) % 1000000000,
          "steps": params['steps'],
          "cfg": params['cfg'],
          "sampler_name": "euler",
          "scheduler": "simple",
          "denoise": params['denoise'],
          "model": ["459:469", 0],
          "positive": ["459:474", 0],
          "negative": ["459:474", 1],
          "latent_image": ["459:468", 0]
        },
        "class_type": "KSampler"
      },
      "459:468": {"inputs": {"switch": False, "on_false": ["459:474", 2], "on_true": ["459:456", 0]}, "class_type": "ComfySwitchNode"},
      "459:469": {"inputs": {"device": "auto", "dtype": "int8", "model": ["459:451", 0]}, "class_type": "QwenImage21Cache"},
      "459:474": {
        "inputs": {
          "prompt": ["459:484", 0],
          "negative_prompt": "" if params['cfg'] <= 1.0 else "blurry, low quality, artifacts, distorted features",
          "resolution": params['resolution'],
          "clip": ["459:453", 0],
          "vae": ["459:454", 0]
        },
        "class_type": "TextEncodeQwenImage21"
      },
      "459:484": {"inputs": {"switch": False, "on_false": params['instruction'], "on_true": params['instruction']}, "class_type": "ComfySwitchNode"}
    }

    # Connect slot nodes deterministically using optimized donor files
    for idx, slot_file in enumerate(optimized_slots, 1):
        node_id = f"slot_node_{idx}"
        workflow[node_id] = {
            "inputs": {"image": slot_file},
            "class_type": "LoadImage"
        }
        workflow["459:474"]["inputs"][f"images.image_{idx}"] = [node_id, 0]

    def event_stream():
        ws = websocket.WebSocket()
        ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")

        req_data = json.dumps({"prompt": workflow, "client_id": client_id}).encode('utf-8')
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_data)
        with urllib.request.urlopen(req) as resp:
            job_meta = json.loads(resp.read())
        job_id = job_meta['prompt_id']
        start_t = time.time()

        while True:
            msg = ws.recv()
            if not isinstance(msg, str):
                continue
            ev = json.loads(msg)
            ev_type = ev.get('type')
            data = ev.get('data', {})

            if ev_type == 'progress':
                val = data.get('value', 0)
                total = data.get('max', params['steps'])
                pct = int((val / total) * 100) if total > 0 else 0
                print(f"\r[Sampling] ⚡ Step {val}/{total} ({pct}%)", end="", flush=True)
                yield f"data: {json.dumps({'type': 'progress', 'value': val, 'max': total})}\n\n"
            elif ev_type == 'executing':
                node = data.get('node')
                if node is None and data.get('prompt_id') == job_id:
                    break
                if node:
                    print(f"\n[Layer] ⚙️ Active Layer: {node}", flush=True)
                    yield f"data: {json.dumps({'type': 'executing', 'node': node})}\n\n"

        ws.close()
        dur = time.time() - start_t

        outs = glob.glob('/content/ComfyUI/output/Qwen_image_2.1*.png')
        latest = max(outs, key=os.path.getctime)
        with open(latest, 'rb') as f:
            b64_str = base64.b64encode(f.read()).decode('utf-8')
        with Image.open(latest) as img:
            is_alpha = img.mode == 'RGBA'

        print(f"\n[Sampling] ✓ Completed in {dur:.1f}s | Output: {os.path.basename(latest)}", flush=True)
        yield f"data: {json.dumps({'type': 'done', 'image_b64': b64_str, 'duration': dur, 'has_alpha': is_alpha})}\n\n"

    return Response(event_stream(), mimetype="text/event-stream")

threading.Thread(target=lambda: app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

print("═" * 65)
print("CONNECTING TO PINGGY HTTPS TUNNEL")
print("═" * 65)

tunnel = pinggy.start_tunnel(forwardto="localhost:5000")

public_url = None
for _ in range(20):
    if hasattr(tunnel, "urls") and tunnel.urls:
        for u in tunnel.urls:
            if u.startswith("https://") and "dashboard" not in u:
                public_url = u
                break
        if public_url:
            break
    time.sleep(1)

if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, rgba(255, 0, 127, 0.15), rgba(121, 40, 202, 0.25), rgba(0, 223, 216, 0.15)); border: 1px solid rgba(255, 255, 255, 0.2); backdrop-filter: blur(20px); border-radius: 20px; padding: 24px; text-align: center; margin: 15px 0; box-shadow: 0 10px 40px rgba(0, 0, 0, 0.5);">
      <div style="margin-bottom: 8px; font-size: 13px; font-weight: 700; color: #f43f5e; font-family: sans-serif;">
        🔴 Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #f43f5e; text-decoration: underline;">AI With Chucky</a>
      </div>
      <h2 style="background: linear-gradient(90deg, #ff007f, #7928ca, #0070f3, #00dfd8); -webkit-background-clip: text; -webkit-text-fill-color: transparent; margin: 0 0 10px 0; font-size: 24px; font-family: sans-serif; font-weight: 800;">Qwen-Image-2.1 Interface is LIVE!</h2>
      <p style="color: #cbd5e1; margin: 0 0 18px 0; font-size: 14px; font-family: sans-serif;">Click below to open the interactive interface:</p>
      <a href="{public_url}" target="_blank" style="background: linear-gradient(90deg, #ff007f, #7928ca, #0070f3, #00dfd8); color: white; padding: 14px 34px; border-radius: 14px; text-decoration: none; font-weight: 800; font-size: 15px; display: inline-block; font-family: sans-serif; box-shadow: 0 4px 25px rgba(255, 0, 127, 0.4); transition: transform 0.2s;">
        👉 Open Interface ({public_url})
      </a>
    </div>
    """))
    print(f"Public URL: {public_url}\n")
else:
    print("⚠️ Tunnel registered URLs:", getattr(tunnel, 'urls', []))

print("═" * 65)
print("LIVE LOG STREAM (Monitoring Task Execution & Sampling Telemetry)")
print("═" * 65 + "\n")

def stream_engine_file():
    with open("/content/engine.log", "r") as f:
        f.seek(0, os.SEEK_END)
        while True:
            l = f.readline()
            if l:
                if any(k in l for k in ["Loading", "Allocated", "Offload", "Error", "Exception", "Traceback", "model_management"]):
                    print(f"[EngineCore] {l.strip()}", flush=True)
            else:
                time.sleep(0.5)

threading.Thread(target=stream_engine_file, daemon=True).start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Execution stopped by user.")